In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import sklearn
sklearn.set_config(display='text')
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier # 앙상블 보팅 알고리즘을 사용하기 위해 import 한다.

앙상블(ensemble)

지도 학습은 피쳐와 레이블을 이용해서 전체 데이터를 분류하는 학습 방법이다.  
지도 학습에 사용되는 각 데이터는 피쳐마다 올바르게 할당된 라벨링 데이터였고 이를 기반으로 학습 알고리즘을 생성하는 과정을 거쳤었다.

앙상블 알고리즘의 핵심 아이디어는 학습 데이터를 기반으로 분류 모델을 여러개 만들고 서로 비교하는 것이다.  
앙상블 학습 과정에서 만든 개별 머신러닝 모델을 분류기(classicfier)라고 하고 여러 개의 분류기를 결합함으로써 개별적인 분류기보다 성능이 뛰어난 최종 분류기를 만드는 것이 앙상블 알고리즘의 목적이다.

보팅(voting)

여러 개의 분류 모델의 결과를 대상으로 투표를 통해서 최종 클래스의 레이블을 결정하는 방법이다.

분류기가 10개 있다고 했을 때, 특정 데이터에 대해서 7개의 분류기는 클래스1이라고 예측하고, 나머지 3개의 분류기는 클래스2라고 예측했을 때 클래스1이 가장 높은 득표수를 보이므로 최종적으로 클래스1로 예측하는 것이다. 이를 다수결 투표라고 하는데 이와 비슷한 방법으로 다수결이 아닌 절반 이상의 분류기의 표를 얻어야 하는 과반수 투표 방식이 있다.

개별 분류기는 지도 학습 방법 중 k-최근접 이웃, 로지스틱 회귀, 나이브 베이즈, 의사결정 트리, 서포트 벡터 머신 등 여러가지 알고리즘을 사용해서 다양한 분류 모델을 만들어 사용할 수 있다.

와인 데이터를 사용해서 와인 종류를 분류하기 위해 데이터를 불러오고 표준화 한다.

In [2]:
# 데이터 불러오기
raw_data = datasets.load_wine() # 사이킷런 라이브러리가 제공하는 와인 데이터를 불러온다.
# print(raw_data)

# 피쳐, 레이블 데이터 저장
xData = raw_data.data # 피쳐 데이터를 저장한다.
yData = raw_data.target # 피쳐 데이터에 따른 레이블을 저장한다.
# print(xData.shape, yData.shape)

# 학습 데이터와 테스트 데이터로 분할
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

# 데이터 표준화(정규화)
scaler = StandardScaler() # 표준화 스케일러 객체를 만든다.
x_train = scaler.fit_transform(x_train) # 학습 데이터를 표준화 스케일러로 표준화하고 적용한다.
x_test = scaler.transform(x_test) # 테스트 데이터를 학습 데이터로 표준화한 스케일러에 적용한다.

모델을 생성하고 학습시킨다.

In [29]:
# 개별 머신러닝 모델 분류기(classicfier)를 만든다.
model_kn = KNeighborsClassifier(n_neighbors=5) # 앙상블 보팅 알고리즘에서 사용할 분류기로 k-최근접 이웃 개별 분류기를 만든다.
model_lr = LogisticRegression(l1_ratio=0.0, C=0.1, solver='saga') # 앙상블 보팅 알고리즘에서 사용할 분류기로 로지스틱 회귀 개별 분류기를 만든다.
model_nb = GaussianNB() # 앙상블 보팅 알고리즘에서 사용할 분류기로 가우시안 나이브 베이즈 개별 분류기를 만든다.
model_dt = DecisionTreeClassifier(max_depth=4, random_state=0) # 앙상블 보팅 알고리즘에서 사용할 분류기로 의사결정 트리 개별 분류기를 만든다.
model_sv = SVC(kernel='rbf', C=0.1, probability=True) # 앙상블 보팅 알고리즘에서 사용할 분류기로 서포트 벡터 머신 개별 분류기를 만든다.

# 위 5개의 개별 분류기를 이용해서 앙상블 보팅 모델을 만든다.
# estimators 속성값으로 앙상블 보팅 모델에서 사용할 개별 분류기를 지정한다.
# voting 속성값으로 투표 방식을 지정한다. 'hard'는 기본값으로 과반수 투표 방식을 'soft'는 다수결 투표 방식을 사용한다.
# 'hard' voting은 확률로 예측하는 predict_proba() 메소드를 사용할 수 없고 'soft' voting은 predict_proba() 메소드를 사용할 수 있다.
# weights 속성값으로 각 개별 분류기에 가중치를 지정해서 가중치 투표를 할 수 있다.
# model = VotingClassifier(estimators=[
    # ('kn', model_kn), ('lr', model_lr), ('nb', model_nb), ('dt', model_dt), ('sv', model_sv)
# ], voting='hard', weights=[1, 1, 1, 1, 1])
# model.fit(x_train, y_train)
model = VotingClassifier(estimators=[
    ('kn', model_kn), ('lr', model_lr), ('nb', model_nb), ('dt', model_dt), ('sv', model_sv)
], voting='soft', weights=[1, 1, 1, 1, 1]).fit(x_train, y_train) # 앙상블 보팅 모델을 만들고 학습시킨다.

학습된 모델로 테스트 데이터를 예측한다.

In [30]:
predict = model.predict(x_test) # predict() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 서포트 벡터 머신 모델을 예측한다.
print(predict)

[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 2 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1]


In [31]:
predict_proba = model.predict_proba(x_test) # predict_proba() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 각 클래스에 속할 확률로 예측한다.
print(predict_proba)

[[9.81904024e-01 1.23315572e-02 5.76441846e-03]
 [4.16135774e-03 6.25244934e-03 9.89586193e-01]
 [3.52987984e-02 9.61875315e-01 2.82588685e-03]
 [9.75855615e-01 1.78889985e-02 6.25538633e-03]
 [1.26856090e-01 8.38062266e-01 3.50816446e-02]
 [1.08258236e-01 8.63686597e-01 2.80551665e-02]
 [9.91536611e-01 4.03914848e-03 4.42424019e-03]
 [3.52387841e-03 1.34948606e-02 9.82981261e-01]
 [8.73101147e-03 9.85641318e-01 5.62767036e-03]
 [4.44343751e-03 9.82141135e-01 1.34154271e-02]
 [1.68888794e-02 2.78486813e-02 9.55262439e-01]
 [4.68520259e-03 1.13111240e-02 9.84003673e-01]
 [9.97143724e-01 1.15226365e-03 1.70401223e-03]
 [2.89817221e-01 7.05611595e-01 4.57118408e-03]
 [7.92868557e-03 6.46443286e-03 9.85606882e-01]
 [2.47619569e-03 9.96702165e-01 8.21639203e-04]
 [9.66625824e-01 2.23382270e-02 1.10359491e-02]
 [9.93394986e-01 2.90276164e-03 3.70225210e-03]
 [1.27129845e-02 3.28909629e-01 6.58377386e-01]
 [9.91755631e-01 6.54336286e-03 1.70100594e-03]
 [3.25904493e-01 6.67708245e-01 6.387262

학습된 모델을 평가한다.

In [33]:
# 혼동 행렬
# confusion_matrix() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 혼동 행렬을 출력한다.
confusion = confusion_matrix(y_test, predict)
print(confusion)

[[16  0  0]
 [ 0 20  1]
 [ 0  0  8]]


In [34]:
# 분류 리포트
# classification_report() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 분류 리포트를 출력한다.
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        16
     class_1       1.00      0.95      0.98        21
     class_2       0.89      1.00      0.94         8

    accuracy                           0.98        45
   macro avg       0.96      0.98      0.97        45
weighted avg       0.98      0.98      0.98        45



In [35]:
# 정확도 평가
# accuracy_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정확도를 계산한다.
accuracy = accuracy_score(y_test, predict)
print(accuracy)

0.9777777777777777


In [36]:
# 정밀도 평가
# precision_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정밀도를 계산한다.
precision = precision_score(y_test, predict, average=None)
print(precision)

[1.         1.         0.88888889]


In [37]:
# 재현율 평가
# recall_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 재현율을 계산한다.
recall = recall_score(y_test, predict, average=None)
print(recall)

[1.         0.95238095 1.        ]


In [38]:
# f1 score 평가
# f1_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 f1 score를 계산한다.
f1 = f1_score(y_test, predict, average=None)
print(f1)

[1.         0.97560976 0.94117647]
